Imports

In [ ]:
import torch
from torch import nn
import numpy as np
import gc
from models import PoinTr
from time import time
import random
from tqdm import tqdm
import matplotlib.pyplot as plt
from pytorch3d.io import load_obj
from pytorch3d.structures import Pointclouds
from pytorch3d.renderer import (
    look_at_view_transform,
    FoVPerspectiveCameras,
    PointsRasterizationSettings,
    PointsRenderer,
    PointsRasterizer,
    AlphaCompositor,
    TexturesVertex,
    RasterizationSettings,
    MeshRenderer,
    MeshRasterizer,
    look_at_view_transform,
    SoftPhongShader,
    PointLights,
    AmbientLights,
    SoftGouraudShader
)

from pytorch3d.ops import sample_farthest_points
from torch.utils.data import Dataset,DataLoader
import pickle

from pytorch3d.io import load_objs_as_meshes
from pytorch3d.structures import Meshes
import torch   
from pytorch3d.ops import sample_points_from_meshes
import os

0. Dataloader Management

In [77]:

class IncompleteDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        # Return the size of the dataset
        return len(self.X)

    def __getitem__(self, idx):
        # Return a single data point and its label
        data_point = self.X[idx]
        label = self.y[idx]
        return data_point, label

# Currently no validation or test set is used, only using data for training...
def create_dataloader(dataset,batch_size = 32, train_ratio=0.9):
 
    train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    #test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    return train_loader,0

def load_dataloader(processed_idx):
    file_path = 'dataloaders/train' + str(processed_idx) + '.pkl'

    with open(file_path, 'rb') as file:
        dataloader_train = pickle.load(file)

    return dataloader_train

def save_dataloader(dataloader,processed_idx,type):
    with open('dataloaders/'+type+str(processed_idx)+'.pkl', 'wb') as f:
        pickle.dump(dataloader, f)
    

def manage_dataloaders(dataset_size,number_of_chunks = 60,batch_size = 4):
    chunk_size = dataset_size // number_of_chunks

    for i in tqdm(range(number_of_chunks)):
        dataset = create_dataset(i*chunk_size,(i+1)*chunk_size)

        train_dataloader, test_dataloader = create_dataloader(dataset,batch_size)
        save_dataloader(train_dataloader,i,"train")
        #save_dataloader(test_dataloader,i,"test")

def create_dataset(it,dataset_size):

    X = []
    y = []

    for i in range(it,dataset_size):
        GT_XYZ = torch.load(os.path.join("target", f'target_{i}.pt')).squeeze(0)
        incomplete = torch.load(os.path.join("incomplete_projections", f'{i}.pt'))

        y.append(GT_XYZ) 
        X.append(incomplete)

    X = torch.stack(X).float()
    y = torch.stack(y).float()

    dataset = IncompleteDataset(X, y)
    return dataset


1. Data Perparation

In [ ]:
def sparsing(dense_tensor, min_keep_ratio=0.3, max_keep_ratio=1, dropout = 1):
    """
    Keeps a random subset of non-zero (object) pixels per [batch, view],
    applying the same mask across all 3 channels, with others set to 0.

    Inputs:
        dense_tensor: [B, V, C, H, W] tensor with RGB data
    Returns:
        sparsified_tensor: same shape, sparsely masked
    """
    
    V, C, H, W = dense_tensor.shape
    sparsified_tensor = torch.full_like(dense_tensor, -1)
    
    #mask_size = np.random.randint(0,2)
    
    views = np.random.choice(4, size=dropout, replace=False)
    #else:
    #    views = np.random.choice(4, size=3, replace=False)

    #print(f'This is views: {views}')

    #view = np.random.randint(0,4)
    #view = -1
    for v in range(V):
            
        #if v == view and np.random.rand() <= dropout:                
        if v in views:
            sparsified_tensor[v] = 0 # broadcasts... masks out one of the inputs
            continue
        
        img = dense_tensor[v]  # [C, H, W]
        # Object mask: any channel non-zero
        object_mask = (img > 0).any(dim=0)  # [H, W]
        num_obj = object_mask.sum().item()
        if num_obj == 0:
            continue
        
        keep_ratio = torch.rand(1).item() * (max_keep_ratio - min_keep_ratio) + min_keep_ratio
        num_keep = int(keep_ratio * num_obj)
        
        # Generate random binary mask for object pixels
        flat_mask = torch.zeros(num_obj, device=img.device)
        if num_keep > 0:
            flat_mask[:num_keep] = 1
            flat_mask = flat_mask[torch.randperm(num_obj)]
        
        # Create and apply the mask
        mask = torch.zeros(H, W, device=img.device)
        mask[object_mask] = flat_mask
        mask = mask.unsqueeze(0).expand(C, -1, -1)  # Expand to [C, H, W]
        
        sparsified_tensor[v] = img * mask
    
    
    return sparsified_tensor

def render_mesh_vertex_colors_normalized(idx,viz=False,shift=45,elevation=15):
    obj_path = "/home/fv1tw4/pytorch3d_test_20250521/meshes/" + str(idx) + ".obj"
    image_size = 256
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    # Load mesh
    verts, faces, _ = load_obj(obj_path)
    faces_idx = faces.verts_idx.to(device)
    verts = verts.to(device)

    # GLOBAL NORMALIZATION
    min_val = verts.min()
    max_val = verts.max()
    verts_normalized = (verts - min_val) / (max_val - min_val + 1e-8)

    # Create vertex-color texture (RGB = globally normalized XYZ)
    textures = TexturesVertex(verts_features=verts_normalized[None])
    mesh = Meshes(verts=[verts], faces=[faces_idx], textures=textures)

    # Configure rasterization
    raster_settings = RasterizationSettings(
        image_size=image_size,
        blur_radius=0.0,
        faces_per_pixel=1,
        bin_size=0
    )

    # Compute bounding sphere center and radius
    center = 0.5 * (verts.max(0).values + verts.min(0).values)
    radius = (verts - center).norm(dim=1).max()

    # Set field of view and compute tight camera distance
    fov = 30.0  # degrees; decrease to zoom in more
    dist = radius.item() / np.sin(np.radians(fov / 2))
    dist *= 1.05  # Add slight margin to avoid clipping

    #shift = 45 # THIS IS CHAGNED SO ITS GIVEN BY ARG.

    # Generate 4 viewpoints
    Rs, Ts = look_at_view_transform(
        dist=dist,
        elev=elevation,
        azim=[0+shift, 90+shift, 180+shift, 270+shift],
        at=center[None].cpu()
    )

    # Ambient-only lighting
    lights = AmbientLights(device=device, ambient_color=((1.0, 1.0, 1.0),))

    all_images = []
    for i in range(4):
        cameras = FoVPerspectiveCameras(
            device=device,
            R=Rs[i][None],
            T=Ts[i][None],
            fov=fov
        )

        renderer = MeshRenderer(
            rasterizer=MeshRasterizer(
                cameras=cameras,
                raster_settings=raster_settings
            ),
            shader=SoftGouraudShader(
                device=device,
                cameras=cameras,
                lights=lights
            )
        )

        # Render
        images = renderer(mesh)
        image_rgba = images[0].cpu().numpy()
        rgb = image_rgba[..., :3]
        alpha = image_rgba[..., 3]

        rgb[alpha == 0] = 0
        rgb = np.clip(rgb, 0, 1)
        
        all_images.append(rgb)
        if viz:
            plt.figure(figsize=(6, 6))
            plt.imshow(rgb)
            plt.title(f"Azimuth: {90*i}°")
            plt.axis('off')
            plt.tight_layout()
            plt.savefig(f"projection_{i}.png")
            plt.show()

    # Save all projections
    all_images = np.array(all_images)
    #np.save(f"projections/{idx}.npy", all_images)
    return all_images


def render_pointcloud_vertex_colors_normalized(verts,idx, viz=True):

    #obj_path = f"/home/fv1tw4/pytorch3d_test_20250521/meshes/{idx}.obj"
    image_size = 256
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

    # Load .obj file (only use verts, ignore faces)
    #verts, _, _ = load_obj(obj_path, load_textures=False)
    verts = verts.to(device)

    # GLOBAL NORMALIZATION
    min_val = verts.min()
    max_val = verts.max()
    verts_normalized = (verts - min_val) / (max_val - min_val + 1e-8)

    # Use normalized XYZ as RGB
    colors = verts_normalized  # (N, 3)
    pointcloud = Pointclouds(points=[verts], features=[colors])

    # Bounding sphere for camera distance
    center = 0.5 * (verts.max(0).values + verts.min(0).values)
    radius = (verts - center).norm(dim=1).max()
    fov = 30.0
    dist = radius.item() / np.sin(np.radians(fov / 2)) * 1.05
    shift = 45

    Rs, Ts = look_at_view_transform(
        dist=dist,
        elev=15,
        azim=[0 + shift, 90 + shift, 180 + shift, 270 + shift],
        at=center[None].cpu(),
    )

    raster_settings = PointsRasterizationSettings(
        image_size=image_size,
        radius=0.01,  # Tune this
        points_per_pixel=10,
    )

    all_images = []
    for i in range(4):
        cameras = FoVPerspectiveCameras(
            device=device, R=Rs[i][None], T=Ts[i][None], fov=fov
        )

        renderer = PointsRenderer(
            rasterizer=PointsRasterizer(cameras=cameras, raster_settings=raster_settings),
            compositor=AlphaCompositor(),
        )

        images = renderer(pointcloud)
        rgb = images[0, ..., :3].cpu().numpy()

        rgb = np.clip(rgb, 0, 1)
        all_images.append(rgb)

        if viz:
            plt.figure(figsize=(6, 6))
            plt.imshow(rgb)
            plt.title(f"Azimuth: {90*i}°")
            plt.axis("off")
            plt.tight_layout()
            plt.savefig(f"projection_{idx}_{i}.png")
            plt.show()

    all_images = np.array(all_images)
    #os.makedirs("incomplete_projections", exist_ok=True)

def load_npy(idx):
    GT_XYZ = np.load("projections/" +str(idx)+".npy")
    return GT_XYZ


def create_pcd(index):
    
    shift = random.randint(0, 360)
    elevation = random.randint(0, 360)
    pcd = render_mesh_vertex_colors_normalized(index,viz=False,shift=shift,elevation=elevation)
    pcd = torch.from_numpy(pcd).to(device)
    pcd = pcd.permute(0, 3, 1, 2)        
    return pcd



def load_pcd_and_fps(device,idx,target_size):
    
    mesh = load_objs_as_meshes([f"meshes/{idx}.obj"], device=device)
    points = sample_points_from_meshes(mesh, target_size)
    return points

def save_pcd(pcd,idx):
    torch.save(pcd,f"incomplete_projections/{idx}.pt")

def proj_2_arr(proj,incomplete_shape_size=4096):
    coords = proj.permute(0, 2, 3, 1).reshape(-1, 3)

    # Filter out (0, 0, 0)
    non_bg = coords[~(coords == 0).all(dim=1)]

    # Get unique coordinates
    unique_coords = torch.unique(non_bg, dim=0)
        
    unique_coords = unique_coords.cuda().unsqueeze(0)
    fps_points, _ = sample_farthest_points(unique_coords, K=incomplete_shape_size)
    fps_points = fps_points.squeeze(0)      # [4096, 3]

    return fps_points


training = 1700
from_idx = 0
to_idx = 1700
# repeating this:
repeat = 4
unique_array = list(range(training*repeat))
random.shuffle(unique_array)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

train = True
test = False

size_of_predictions = 8192
if train:
    for idx in tqdm(range(from_idx, to_idx)):
        target_pcd = load_pcd_and_fps(device,idx,target_size = size_of_predictions)
        
        pcd = create_pcd(idx)
        pcd_sparse1 = sparsing(pcd,dropout=1,min_keep_ratio=1,max_keep_ratio=1)
        
        pcd = create_pcd(idx)
        pcd_sparse2 = sparsing(pcd,dropout=2,min_keep_ratio=1,max_keep_ratio=1)
        
        pcd = create_pcd(idx)
        pcd_sparse3 = sparsing(pcd,dropout=2,min_keep_ratio=1,max_keep_ratio=1)
        
        pcd = create_pcd(idx)
        pcd_sparse_extra = sparsing(pcd,dropout=2,min_keep_ratio=1,max_keep_ratio=1)
        
        id1 = unique_array.pop()
        id2 = unique_array.pop()
        id3 = unique_array.pop()
        id4 = unique_array.pop()

        # Needs to be saved....
        save_pcd(proj_2_arr(pcd_sparse1),id1)
        save_pcd(proj_2_arr(pcd_sparse2), id2)
        save_pcd(proj_2_arr(pcd_sparse3), id3)
        save_pcd(proj_2_arr(pcd_sparse_extra),id4)
        
        torch.save(target_pcd, f"target/target_{id1}.pt")
        torch.save(target_pcd, f"target/target_{id2}.pt")
        torch.save(target_pcd, f"target/target_{id3}.pt")
        torch.save(target_pcd, f"target/target_{id4}.pt")


100%|██████████| 1700/1700 [1:31:27<00:00,  3.23s/it]  


2. Training Loop

In [ ]:
class Config:
    trans_dim = 128      # Transformer embedding dimension
    knn_layer = 1        # KNN layers in PCTransformer
    num_pred = 8096//2     # number of fine points to generate
    num_query = 256     # number of query (coarse) points
    num_heads = 4


In [83]:
from pytorch3d.loss import chamfer_distance
#from geomloss import SamplesLoss

#sinkhorn = SamplesLoss("sinkhorn", p=2, blur=0.05)

def cd_loss(pc1,pc2,norm=2):
    if norm == 1: #L1
        loss,_ = chamfer_distance(pc1,pc2,norm=1)
    else: # L2
        loss,_ = chamfer_distance(pc1,pc2,norm=2)
    return loss

def emd(pc1,pc2): #L2
    pc1 = pc1.contiguous()
    pc2 = pc2.contiguous()
    
    loss = sinkhorn(pc1,pc2)
    return loss.mean()


def combined_loss(pred,target):
    return cd_loss(pred,target,norm=1)


    

In [135]:

def validation(model,device,validation_dataloader):
    val_loss = 0
    iter = 0
    model.eval()
    with torch.no_grad():
        for x,y in validation_dataloader:
            x, y = x.to(device), y.to(device)
            
            pred_y = model(x)[1]
   
            loss = combined_loss(pred_y, y)
            val_loss += loss.item()
            iter +=1

    return val_loss/iter

import torch

# --- 1️⃣ Load pretrained checkpoint ---
def load_pretrained_and_freeze():
    ckpt_path = "checkpoints/pretrained_shapenet55.pth"
    ckpt = torch.load(ckpt_path, map_location="cpu")
    pretrained_dict = ckpt["model"]

    # --- 2️⃣ Initialize your model ---
    config = Config()
    model = PoinTr(config).cuda()
    model_dict = model.state_dict()

    # --- 3️⃣ Filter for self-attention weights only ---
    attn_weights = {
        k.replace("module.", ""): v
        for k, v in pretrained_dict.items()
        if "attn" in k  # matches all self-attention layer parameters
    }
    print(f"✅ Found {len(attn_weights)} attention parameters to load.")

    # --- 4️⃣ Load those weights into your model ---
    model_dict.update(attn_weights)
    model.load_state_dict(model_dict, strict=False)
    print("✅ Loaded all self-attention weights successfully.")

    # --- 5️⃣ Freeze attention parameters so they won't train ---
    for name, param in model.named_parameters():
        if "attn" in name:
            param.requires_grad = False
            print(f"🔒 Frozen {name}")

    # --- 6️⃣ Sanity check ---
    frozen = [n for n, p in model.named_parameters() if not p.requires_grad]
    print(f"\n🔒 Total frozen parameters: {len(frozen)}")
    print("✅ Ready for fine-tuning (attention layers frozen).")
    return model
def train(name,dataset_size,number_of_chunks = 60,learning_rate=0.001,batch_size=4,epochs = 50,load_checkpoint="",val_chunks = 1):
    torch.set_float32_matmul_precision('high')

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f'Device is: {device}')
    #config = Config()
    #model = PoinTr(config)
    model = load_pretrained_and_freeze()
    opt = torch.optim.Adam(model.parameters(), lr = learning_rate)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='min', 
                                                       factor=0.1, patience=5, 
                                                       verbose=False,min_lr=1e-4)
    torch.backends.cudnn.benchmark = True
    model.to(device)

    losses = []            
    val_losses = []
    best_loss = 1e8
    if load_checkpoint != "":
        checkpoint = torch.load(load_checkpoint, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        opt.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        losses = checkpoint['losses']
        val_losses = checkpoint['val_loss']
    else:
        start_epoch = 1

    for epoch in range(start_epoch,epochs+1):
        print("-----------------------------------------------------------")
        print(f'This is Epoch: {epoch}/{epochs}...')
        model.train()
        stime = time()
        chunk_loss = 0
        temp_val_losses = 0
        for i in range(0, number_of_chunks):
            train_dataloader = load_dataloader(i)

            if i % 5 == 0:
                gc.collect()
                torch.cuda.empty_cache()

            if i >= number_of_chunks-val_chunks: 
                temp_val_losses += validation(model,device,train_dataloader)
                continue

            loader_loss = 0
            total = 0

            for it_, (x, y) in enumerate(train_dataloader):
                x, y = x.to(device), y.to(device)
                pred_y = model(x)[1]
                opt.zero_grad()

                loss = combined_loss(y, pred_y)
                loss.backward()
                #torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) shoud I?.....
                opt.step()
                
                loader_loss += loss.item()

                total += 1

            chunk_loss += loader_loss/total
            
            #chunk_d_loss += d_loss_track/total
            #print(f"Chunk [{i+1}/{number_of_chunks}] - Training Loss: {loader_loss/total:.4f}")
        
        losses.append(chunk_loss/number_of_chunks)
        val_losses.append(temp_val_losses/val_chunks)

        scheduler.step(temp_val_losses)
        temp_val_losses = 0

        ftime = time()
        print(f"Epoch [{epoch}/{epochs}] trained in {ftime - stime}s; Training loss => {losses[-1]:.4f} - Validation Loss: {val_losses[-1]:.4f}")
        print(f'learning_rate: {scheduler.get_last_lr()}')

        if epoch % 5 == 0 or epoch == 1:
            update_loss_png(losses, val_losses,epoch, name)
            
        if val_losses[-1] < best_loss:
            best_loss = val_losses[-1]
        
            checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': opt.state_dict(),
            'losses': losses,
            'val_loss': val_losses
            }
            torch.save(checkpoint, "checkpoints/" + name + ".pth")
    
    checkpoint = {
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': opt.state_dict(),
    'losses': losses,
    'val_loss': val_losses
    }
    torch.save(checkpoint, "checkpoints/" + name + "_final.pth")


def update_loss_png(losses, val_loss,epoch, name):

    plt.figure(figsize=(12, 6))
    plt.plot(losses, label='Training Loss', alpha=0.4)    
    plt.plot(val_loss, label='Validation Loss', alpha=0.4)


    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Training Loss and Validation Loss')

    # Save the plot
    plt.savefig(f"losses_pngs/{name}.png") # Something else comes here..............................
    plt.close()

def setup_and_train(name,setup=True, checkpoint=""):
    batch = 8
    dataset_size = 2000 #the wjole dataset size is: 1464, for now...
    number_of_chunks = 16 # (1230*13/16)/1230 --> 81%
    val_chunks=3
    if setup:
        manage_dataloaders(dataset_size,number_of_chunks,batch)
    train(name,dataset_size,number_of_chunks,batch_size=batch,epochs=1_000,learning_rate=1e-3,load_checkpoint=checkpoint,val_chunks=val_chunks)


4. Training parameters and starting the training:

In [136]:
class Config:
    trans_dim = 384      # Transformer embedding dimension
    knn_layer = 8        # KNN layers in PCTransformer
    num_pred = 8096//2     # number of fine points to generate
    num_query = 256     # number of query (coarse) points
    num_heads = 4
setup_and_train("Pointr_v1",setup=False)

Device is: cuda


/tmp/ipykernel_1651/2090782173.py:22: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")


 Transformer with knn_layer 8.
✅ Found 82 attention parameters to load.
✅ Loaded all self-attention weights successfully.
🔒 Frozen base_model.encoder.0.attn.qkv.weight
🔒 Frozen base_model.encoder.0.attn.proj.weight
🔒 Frozen base_model.encoder.0.attn.proj.bias
🔒 Frozen base_model.encoder.1.attn.qkv.weight
🔒 Frozen base_model.encoder.1.attn.proj.weight
🔒 Frozen base_model.encoder.1.attn.proj.bias
🔒 Frozen base_model.encoder.2.attn.qkv.weight
🔒 Frozen base_model.encoder.2.attn.proj.weight
🔒 Frozen base_model.encoder.2.attn.proj.bias
🔒 Frozen base_model.encoder.3.attn.qkv.weight
🔒 Frozen base_model.encoder.3.attn.proj.weight
🔒 Frozen base_model.encoder.3.attn.proj.bias
🔒 Frozen base_model.encoder.4.attn.qkv.weight
🔒 Frozen base_model.encoder.4.attn.proj.weight
🔒 Frozen base_model.encoder.4.attn.proj.bias
🔒 Frozen base_model.encoder.5.attn.qkv.weight
🔒 Frozen base_model.encoder.5.attn.proj.weight
🔒 Frozen base_model.encoder.5.attn.proj.bias
🔒 Frozen base_model.decoder.0.self_attn.qkv.weight

/home/fv1tw4/pytorch3d_env/lib/python3.8/site-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch [1/1000] trained in 95.42920398712158s; Training loss => 0.5678 - Validation Loss: 0.6500
learning_rate: [0.001]
-----------------------------------------------------------
This is Epoch: 2/1000...
Epoch [2/1000] trained in 93.80334162712097s; Training loss => 0.5297 - Validation Loss: 0.6516
learning_rate: [0.001]
-----------------------------------------------------------
This is Epoch: 3/1000...
Epoch [3/1000] trained in 92.0979654788971s; Training loss => 0.5337 - Validation Loss: 0.6529
learning_rate: [0.001]
-----------------------------------------------------------
This is Epoch: 4/1000...
Epoch [4/1000] trained in 90.63445472717285s; Training loss => 0.5444 - Validation Loss: 0.9653
learning_rate: [0.001]
-----------------------------------------------------------
This is Epoch: 5/1000...
Epoch [5/1000] trained in 88.514728307724s; Training loss => 0.5331 - Validation Loss: 0.6533
learning_rate: [0.001]
-----------------------------------------------------------
This is 

KeyboardInterrupt: 

In [121]:
pretrained = torch.load("checkpoints/pretrained_shapenet55.pth")

/tmp/ipykernel_1651/2930616668.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrained = torch.load("checkpoints/pretrained_shapenet55.pth")


In [137]:
state_dict = pretrained['model']
for k in state_dict.keys():
    print(k)

module.base_model.grouper.input_trans.weight
module.base_model.grouper.input_trans.bias
module.base_model.grouper.layer1.0.weight
module.base_model.grouper.layer1.1.weight
module.base_model.grouper.layer1.1.bias
module.base_model.grouper.layer2.0.weight
module.base_model.grouper.layer2.1.weight
module.base_model.grouper.layer2.1.bias
module.base_model.grouper.layer3.0.weight
module.base_model.grouper.layer3.1.weight
module.base_model.grouper.layer3.1.bias
module.base_model.grouper.layer4.0.weight
module.base_model.grouper.layer4.1.weight
module.base_model.grouper.layer4.1.bias
module.base_model.pos_embed.0.weight
module.base_model.pos_embed.0.bias
module.base_model.pos_embed.1.weight
module.base_model.pos_embed.1.bias
module.base_model.pos_embed.1.running_mean
module.base_model.pos_embed.1.running_var
module.base_model.pos_embed.1.num_batches_tracked
module.base_model.pos_embed.3.weight
module.base_model.pos_embed.3.bias
module.base_model.input_proj.0.weight
module.base_model.input_pro